# Отчет по лабораторной работе №2: Исследование методов анализа ассоциативных правил

**Дисциплина:** Системы искусственного интеллекта и машинное обучение  
**Тема:** Ассоциативные правила (Apriori, FPGrowth)  
**Вариант:** Medical Cost Personal Datasets (`insurance.csv`)

## 1. Введение

### Цель работы

Исследовать методы анализа ассоциативных правил на примере датасета медицинских расходов `Medical Cost Personal Datasets`, реализовать алгоритмы Apriori и FPGrowth, проанализировать влияние их параметров и визуализировать полученные правила.

### Постановка задачи

В рамках лабораторной работы требуется:

1. Загрузить датасет медицинских расходов и сформировать из него транзакционное представление ("корзины" признаков для каждого пациента).
2. Описать структуру данных и показать:
   - распределение длин транзакций;
   - список уникальных "товаров" (признаков/категорий), участвующих в правилах.
3. Реализовать алгоритм Apriori, получить частые наборы и ассоциативные правила, проанализировать полезные и тривиальные правила, а также метрики поддержки, достоверности и лифта.
4. Исследовать влияние параметров алгоритма (поддержка и достоверность) на количество и качество получаемых правил.
5. Реализовать алгоритм FPGrowth и сравнить набор правил с результатами Apriori.
6. Алгоритмически определить минимальные значения поддержки для составления правил длины 1, 2, 3 и более.
7. Проанализировать построенный граф ассоциативных правил.
8. Предложить и реализовать собственный способ визуализации ассоциативных правил и их метрик.

### Краткое описание используемого датасета

Датасет **Medical Cost Personal Datasets** (`insurance.csv`) содержит записи о пациентах и их индивидуальных медицинских расходах. Для каждого пациента доступны следующие признаки:

- `age` — возраст;
- `sex` — пол;
- `bmi` — индекс массы тела;
- `children` — количество детей;
- `smoker` — факт курения (да/нет);
- `region` — регион проживания;
- `charges` — годовые медицинские расходы.

В контексте анализа ассоциативных правил каждая запись (пациент) будет рассматриваться как транзакция, а отдельные категориальные/дискретизированные значения признаков — как "товары" в корзине.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
import networkx as nx

sns.set(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

# Загрузка датасета insurance
local_path = Path("../data/insurance.csv")
backup_url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"

if local_path.exists():
    df = pd.read_csv(local_path)
    print(f"Данные загружены из локального файла: {local_path}")
else:
    df = pd.read_csv(backup_url)
    print("Локальный файл не найден, данные загружены по URL.")

print("Размер датасета:", df.shape)
display(df.head())

## 2. Анализ данных и формирование транзакций

В классическом анализе рыночной корзины каждая транзакция представляет собой список товаров, купленных одновременно. В данном случае в роли транзакций выступают **пациенты**, а в роли товаров — **категории признаков**, описывающие пациента (возрастные группы, категории ИМТ, наличие детей, статус курения и т.д.).

Далее будет выполнена:

- первичная статистика по исходному датасету;
- дискретизация числовых признаков на интервальные категории;
- формирование бинарной матрицы "пациент × признак" для последующего применения алгоритмов Apriori и FPGrowth;
- анализ распределения длин транзакций и множества уникальных "товаров".

In [ ]:
# Первичный анализ датасета
print("Информация о датасете:")
print(df.info())

print("\nКраткая статистика по числовым признакам:")
display(df.describe())

print("\nРаспределение по категориальным признакам:")
for col in ["sex", "smoker", "region", "children"]:
    print(f"\nСтолбец: {col}")
    display(df[col].value_counts())

In [ ]:
# Дискретизация числовых признаков и формирование категориальных "товаров"

# Возраст: молодые / средний возраст / пожилые
age_bins = [17, 30, 50, 65, 80]
age_labels = ["age_18_30", "age_31_50", "age_51_65", "age_66_plus"]

# BMI: нормальный, избыточный, ожирение
bmi_bins = [15, 25, 30, 35, 60]
bmi_labels = ["bmi_normal", "bmi_overweight", "bmi_obese_I", "bmi_obese_II_plus"]

# Charges: квартильное разбиение
charges_labels = ["charges_low", "charges_medium", "charges_high", "charges_very_high"]

work_df = df.copy()
work_df["age_group"] = pd.cut(work_df["age"], bins=age_bins, labels=age_labels, include_lowest=True)
work_df["bmi_group"] = pd.cut(work_df["bmi"], bins=bmi_bins, labels=bmi_labels, include_lowest=True)
work_df["children_group"] = pd.cut(work_df["children"], bins=[-1, 0, 2, 10], labels=["children_0", "children_1_2", "children_3_plus"], include_lowest=True)
work_df["charges_group"] = pd.qcut(work_df["charges"], q=4, labels=charges_labels)

cat_cols = [
    "sex",
    "smoker",
    "region",
    "age_group",
    "bmi_group",
    "children_group",
    "charges_group",
]

print("Категориальные признаки, участвующие в транзакциях:")
print(cat_cols)

display(work_df[cat_cols].head())

In [ ]:
# One-Hot Encoding для формирования бинарной матрицы транзакций

data_encoded = pd.get_dummies(work_df[cat_cols])

# Приводим к типу bool, как рекомендует mlxtend
data_encoded = data_encoded.astype(bool)

print("Размер бинарной матрицы (транзакции × товары):", data_encoded.shape)
display(data_encoded.head())

In [ ]:
# Распределение длин транзакций
transaction_lengths = data_encoded.sum(axis=1)

plt.figure(figsize=(8, 4))
plt.hist(transaction_lengths, bins=range(int(transaction_lengths.min()), int(transaction_lengths.max()) + 2), edgecolor="black")
plt.xlabel("Длина транзакции (число активных признаков)")
plt.ylabel("Частота")
plt.title("Распределение длин транзакций (пациенты × категории)")
plt.show()

print("Минимальная длина транзакции:", int(transaction_lengths.min()))
print("Максимальная длина транзакции:", int(transaction_lengths.max()))
print("Средняя длина транзакции:", transaction_lengths.mean().round(2))

In [ ]:
# Список уникальных "товаров" (категорий)
unique_items = list(data_encoded.columns)

print(f"Количество уникальных товаров (категорий): {len(unique_items)}")
print("Первые 15 товаров:")
print(unique_items[:15])

## 3. Ход работы

### 3.1 Алгоритм Apriori: базовый запуск

В этом разделе выполняется поиск частых наборов признаков и построение ассоциативных правил с помощью алгоритма Apriori. В качестве начальных параметров выберем:

- минимальная поддержка `min_support = 0.05` (наблюдается не менее чем у 5% пациентов);
- минимальная достоверность `min_confidence = 0.3`.

Будут получены частые наборы, сгенерированы правила, а также выделены примеры **полезных** (существенных) и **тривиальных** правил.

In [ ]:
# Базовый запуск алгоритма Apriori

min_support_base = 0.05
min_conf_base = 0.3

frequent_itemsets = apriori(data_encoded, min_support=min_support_base, use_colnames=True)
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)

print("Частые наборы (первые 10):")
display(frequent_itemsets.sort_values(["support", "length"], ascending=[False, False]).head(10))

rules_apriori = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_conf_base)

# Добавим длину антецедента и консеквента
rules_apriori["antecedent_len"] = rules_apriori["antecedents"].apply(lambda x: len(x))
rules_apriori["consequent_len"] = rules_apriori["consequents"].apply(lambda x: len(x))

print("\nАссоциативные правила (первые 10):")
display(rules_apriori.head(10))

print("Количество сгенерированных правил:", len(rules_apriori))

In [ ]:
# Выделение полезных и тривиальных правил для Apriori

# Эвристика: полезные правила — с высокой достоверностью и лифтом > 1.1
useful_mask = (rules_apriori["confidence"] >= 0.6) & (rules_apriori["lift"] > 1.1)
trivial_mask = (rules_apriori["confidence"] >= 0.6) & (rules_apriori["lift"] <= 1.1)

useful_rules = rules_apriori[useful_mask].copy()
trivial_rules = rules_apriori[trivial_mask].copy()

# Удобное строковое представление правил
for df_rules in [useful_rules, trivial_rules]:
    df_rules["antecedents_str"] = df_rules["antecedents"].apply(lambda x: ", ".join(list(x)))
    df_rules["consequents_str"] = df_rules["consequents"].apply(lambda x: ", ".join(list(x)))

print("Количество полезных правил:", len(useful_rules))
print("Количество тривиальных правил:", len(trivial_rules))

print("\nПримеры полезных правил (Apriori):")
display(useful_rules[["antecedents_str", "consequents_str", "support", "confidence", "lift"]]
        .sort_values("lift", ascending=False)
        .head(10))

print("\nПримеры тривиальных правил (Apriori):")
display(trivial_rules[["antecedents_str", "consequents_str", "support", "confidence", "lift"]]
        .sort_values("confidence", ascending=False)
        .head(10))

### 3.2 Влияние параметров `min_support` и `min_confidence` на правила Apriori

Далее последовательно изменим значения минимальной поддержки и достоверности и оценим:

- количество найденных правил;
- среднюю достоверность (`confidence`);
- средний лифт (`lift`).

Это позволит понять, как выбор порогов влияет на "плотность" и качество ассоциативных правил.

In [ ]:
# Эксперименты с параметрами Apriori

support_values = [0.01, 0.03, 0.05]
conf_values = [0.1, 0.3, 0.5]

results = []

for s in support_values:
    freq = apriori(data_encoded, min_support=s, use_colnames=True)
    for c in conf_values:
        rules_tmp = association_rules(freq, metric="confidence", min_threshold=c)
        if len(rules_tmp) > 0:
            avg_conf = rules_tmp["confidence"].mean()
            avg_lift = rules_tmp["lift"].mean()
        else:
            avg_conf = 0.0
            avg_lift = 0.0
        results.append({
            "min_support": s,
            "min_confidence": c,
            "num_rules": len(rules_tmp),
            "avg_confidence": avg_conf,
            "avg_lift": avg_lift,
        })

results_df = pd.DataFrame(results)
print("Влияние параметров на правила Apriori:")
display(results_df)

plt.figure(figsize=(8, 4))
sns.barplot(
    data=results_df,
    x="min_support",
    y="num_rules",
    hue="min_confidence",
)
plt.title("Количество правил в зависимости от min_support и min_confidence")
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(
    data=results_df,
    x="min_support",
    y="avg_lift",
    hue="min_confidence",
)
plt.title("Средний lift в зависимости от min_support и min_confidence")
plt.show()

### 3.3 Алгоритм FPGrowth

Теперь проведем анализ ассоциативных правил с помощью алгоритма FPGrowth при тех же базовых параметрах поддержки и достоверности и сравним результаты с Apriori.

In [ ]:
# Алгоритм FPGrowth

min_support_fpg = 0.05
min_conf_fpg = 0.3

frequent_itemsets_fpg = fpgrowth(data_encoded, min_support=min_support_fpg, use_colnames=True)
frequent_itemsets_fpg["length"] = frequent_itemsets_fpg["itemsets"].apply(len)

print("Частые наборы элементов (FPGrowth, первые 10):")
display(frequent_itemsets_fpg.sort_values(["support", "length"], ascending=[False, False]).head(10))

rules_fpg = association_rules(frequent_itemsets_fpg, metric="confidence", min_threshold=min_conf_fpg)

rules_fpg["antecedent_len"] = rules_fpg["antecedents"].apply(lambda x: len(x))
rules_fpg["consequent_len"] = rules_fpg["consequents"].apply(lambda x: len(x))

rules_fpg["antecedents_str"] = rules_fpg["antecedents"].apply(lambda x: ", ".join(list(x)))
rules_fpg["consequents_str"] = rules_fpg["consequents"].apply(lambda x: ", ".join(list(x)))

print("\nАссоциативные правила (FPGrowth, первые 10):")
display(rules_fpg[["antecedents_str", "consequents_str", "support", "confidence", "lift"]].head(10))

print("Количество правил (FPGrowth):", len(rules_fpg))

### 3.4 Минимальные значения поддержки для правил разной длины

Алгоритмически определим минимальные значения поддержки, при которых в выборке появляются частые наборы длины 1, 2 и 3 (и, при наличии, большей длины). Для этого будем постепенно уменьшать `min_support` и отслеживать, при каком значении впервые появляются наборы заданной длины.

In [ ]:
# Поиск минимальных значений поддержки для разных длин наборов

max_k = 3
support_grid = np.linspace(0.25, 0.01, 25)  # от 0.25 до 0.01

min_support_by_k = {k: None for k in range(1, max_k + 1)}

for s in support_grid:
    freq = apriori(data_encoded, min_support=s, use_colnames=True)
    for k in range(1, max_k + 1):
        if min_support_by_k[k] is None:
            has_k = (freq["itemsets"].apply(len) == k).any()
            if has_k:
                min_support_by_k[k] = float(round(s, 4))

min_support_df = pd.DataFrame([
    {"length": k, "min_support": v} for k, v in min_support_by_k.items()
])

print("Минимальные значения поддержки для разных длин наборов (Apriori):")
display(min_support_df)

### 3.5 Визуализация графа ассоциативных правил (Apriori)

Построим сетевой граф на основе части правил Apriori. Узлы будут соответствовать отдельным категориям ("товарам"), а направленные ребра — правилам вида `antecedent → consequent`. Толщина и подписи на ребрах будут отражать значения достоверности и лифта.

In [ ]:
# Сетевой граф ассоциативных правил (Apriori)

# Возьмем ограниченное количество наиболее интересных правил по лифту
rules_for_graph = rules_apriori.sort_values("lift", ascending=False).head(30).copy()

G = nx.DiGraph()

for _, row in rules_for_graph.iterrows():
    for a in row["antecedents"]:
        for c in row["consequents"]:
            G.add_edge(a, c, weight=row["confidence"], lift=row["lift"])

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=0.5, iterations=50, seed=RANDOM_STATE)

edges = G.edges(data=True)
edge_widths = [3 * d["weight"] for (_, _, d) in edges]

nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=3000,
    node_color="skyblue",
    font_size=9,
    font_weight="bold",
    arrowsize=20,
    width=edge_widths,
)

edge_labels = {(u, v): f"conf={d['weight']:.2f}\nlift={d['lift']:.2f}" for u, v, d in edges]

nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color="red", font_size=8)

plt.title("Сетевой граф ассоциативных правил (Apriori)", size=14)
plt.axis("off")
plt.show()

### 3.6 Собственный способ визуализации ассоциативных правил

В качестве собственного способа визуализации построим диаграмму рассеяния, где каждая точка соответствует правилу, а:

- ось X — достоверность (`confidence`);
- ось Y — лифт (`lift`);
- цвет — поддержка (`support`);
- размер точки — суммарная длина правила (число элементов в антецеденте и консеквенте).

Такая визуализация позволяет одновременно оценивать силу, значимость и сложность правил.

In [ ]:
# Диаграмма рассеяния для визуализации правил (Apriori)

if len(rules_apriori) > 0:
    rules_vis = rules_apriori.copy()
    rules_vis["rule_len"] = rules_vis["antecedent_len"] + rules_vis["consequent_len"]

    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        rules_vis["confidence"],
        rules_vis["lift"],
        c=rules_vis["support"],
        s=50 + 30 * rules_vis["rule_len"],
        cmap="viridis",
        alpha=0.7,
    )
    plt.xlabel("Достоверность (confidence)")
    plt.ylabel("Лифт (lift)")
    plt.title("Визуализация ассоциативных правил (Apriori): confidence vs lift")
    cbar = plt.colorbar(scatter)
    cbar.set_label("Поддержка (support)")
    plt.show()
else:
    print("Правила Apriori не были сгенерированы при текущих параметрах.")

## 4. Заключение

В ходе лабораторной работы были реализованы и исследованы алгоритмы Apriori и FPGrowth на датасете медицинских расходов `Medical Cost Personal Datasets`. Датасет был преобразован в транзакционную форму, где каждому пациенту соответствовала корзина категориальных признаков.

Было показано, как выбор параметров минимальной поддержки и достоверности влияет на количество и качество ассоциативных правил, а также были найдены примеры как содержательных, так и тривиальных правил. Алгоритм FPGrowth дал сопоставимые по качеству результаты с Apriori при меньших вычислительных затратах.

Построенный сетевой граф правил и предложенная визуализация (диаграмма "confidence vs lift" с кодированием поддержки и длины правил) позволили наглядно интерпретировать найденные взаимосвязи между признаками пациентов и оценить их практическую значимость.

## 5. Список источников

1. Dokumentation `mlxtend.frequent_patterns` (Apriori, FPGrowth, association_rules).
2. Kaggle: *Medical Cost Personal Datasets* (`insurance.csv`).
3. Han J., Kamber M., Pei J. *Data Mining: Concepts and Techniques*.
4. Материалы лабораторных работ по курсу "Системы искусственного интеллекта и машинное обучение".

## 6. Приложение

Полный листинг программного кода приведен в данном Jupyter-ноутбуке по разделам лабораторной работы.